In [ ]:
import os

os.environ['KAGGLE_USERNAME'] = "xxxxxx"
os.environ['KAGGLE_KEY'] = "xxxxxx"

partial_data = True #set to True to use a subset of the dataset
temporal = False #set to True to see the results using the User-Centered Temporal Split of the training and test data


Download and unzip comments and articles

In [ ]:
#Download comments data
if partial_data:
  !kaggle datasets download -d benjaminawd/new-york-times-articles-comments-2020 -f nyt-comments-part0.csv
  !unzip -o nyt-comments-part0.csv

  !kaggle datasets download -d benjaminawd/new-york-times-articles-comments-2020 -f nyt-comments-part1.csv
  !unzip -o nyt-comments-part1.csv

else:
  !kaggle datasets download -d benjaminawd/new-york-times-articles-comments-2020 -f nyt-comments-2020.csv
  !unzip -o nyt-comments-2020.csv
#Download article data
!kaggle datasets download -d benjaminawd/new-york-times-articles-comments-2020 -f nyt-articles-2020.csv
!unzip -o nyt-articles-2020.csv

Dataset URL: https://www.kaggle.com/datasets/benjaminawd/new-york-times-articles-comments-2020
License(s): CC-BY-NC-SA-4.0
100% 97.8M/97.8M [00:01<00:00, 81.0MB/s]

Archive:  nyt-comments-part0.csv.zip
  inflating: nyt-comments-part0.csv  
Dataset URL: https://www.kaggle.com/datasets/benjaminawd/new-york-times-articles-comments-2020
License(s): CC-BY-NC-SA-4.0
100% 97.6M/97.6M [00:00<00:00, 116MB/s]

Archive:  nyt-comments-part1.csv.zip
  inflating: nyt-comments-part1.csv  
Dataset URL: https://www.kaggle.com/datasets/benjaminawd/new-york-times-articles-comments-2020
License(s): CC-BY-NC-SA-4.0
100% 2.91M/2.91M [00:00<00:00, 36.6MB/s]

Archive:  nyt-articles-2020.csv.zip
  inflating: nyt-articles-2020.csv   


PySpark initialization

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
sc = spark.sparkContext

Read comments via spark.read.csv, gets dataframe as output

In [ ]:
if partial_data:
  comments_df = spark.read.csv('nyt-comments-part0.csv', header=True, inferSchema=True, multiLine=True, quote='"', sep=',', escape='"')
  comments_df = comments_df.union(spark.read.csv('nyt-comments-part1.csv', header=True, inferSchema=True, multiLine=True, quote='"', sep=',', escape='"'))
else:
  comments_df = spark.read.csv('nyt-comments-2020.csv', header=True, inferSchema=True, multiLine=True, quote='"', sep=',', escape='"')

  comments_df.show(10)
  comments_df.count()

Create a map articleID(string) -> articleID(int) to convert the links of articles to integer IDs

In [ ]:
article_IDs_rdd = comments_df.select('articleID').distinct().rdd.map(lambda x: (x[0]))
article_IDs_map = sc.broadcast(article_IDs_rdd.zipWithIndex().collectAsMap())

#reverse_map = {v: k for k, v in article_IDs_map.value.items()}
#article_link = sc.broadcast(reverse_map)

Convert the comments df to a comments RDD, after selecting the userID, articleID and createDate columns, we map also the articleID to the integer ID

In [ ]:
comments_rdd = comments_df.select(['userID','articleID','createDate']).rdd.map(lambda x: ((x[0], article_IDs_map.value[x[1]]), x[2])).cache()
comments_rdd.count()

1000000

Keep only the oldest comment date per pair (userID, articleID), we ignore the answers to other comments and discussions

In [ ]:

comments_article_singleTime_rdd = comments_rdd.reduceByKey(lambda date1,date2: min(date1,date2))
comments_rdd.unpersist()

#comments_article_singleTime_rdd.takeOrdered(10)

PythonRDD[31] at RDD at PythonRDD.scala:56

RDD containing the number of articles the user has commented on per user



In [ ]:
user_comments_count = comments_article_singleTime_rdd.map(lambda x: (x[0][0],1)).reduceByKey(lambda x,y: x+y)

#user_comments_count.takeOrdered(10)

users that have more than 9 comments, this will be used later in a join with the comments table to keep only the rows of the users that have commented more than 9 times (avoids user cold-start)

In [ ]:
comments_over_10_rdd = user_comments_count.filter(lambda x: x[1] >= 10)

#comments_over_10_rdd.take(10)


In [ ]:

print('there were {} users, now there are {} users'.format(user_comments_count.count(), comments_over_10_rdd.count()))

there were 136999 users, now there are 15123 users


remaps the comments_article rdd to (userID, articleID, createDate)

In [ ]:
remapped_comments_article_singleTime = comments_article_singleTime_rdd.map(lambda x: ((x[0][0]),(x[0][1], x[1]))).cache()

remapped_comments_article_singleTime.take(5)


[(60215558, (620, datetime.datetime(2020, 1, 1, 1, 5, 46))),
 (65691034, (620, datetime.datetime(2020, 1, 1, 1, 52, 25))),
 (73299044, (620, datetime.datetime(2020, 1, 1, 1, 54, 26))),
 (85853572, (620, datetime.datetime(2020, 1, 1, 4, 12, 42))),
 (102161210, (620, datetime.datetime(2020, 1, 1, 5, 23, 55)))]

Remove rows with users that have commented in less than 10 articles

In [ ]:

filtered_comments_article_singleTime = remapped_comments_article_singleTime.join(comments_over_10_rdd).map(lambda x: (x[0], x[1][0])).cache()

#filtered_comments_article_singleTime.takeOrdered(10)

In [ ]:

print('there were {} rows, now there are {} rows'.format(remapped_comments_article_singleTime.count(), filtered_comments_article_singleTime.count()))

there were 703088 rows, now there are 449892 rows


SPLIT OF DATASET


In [ ]:
def orderedDateList(iterator):
  return sorted(iterator, key=lambda x: x[1])

For the temporal split, group and transform the comments into (user, orderedListOf((articleID, createDate)) by createDate (first items are the oldest)

In [ ]:
if temporal:
  ordered_comments_by_date = filtered_comments_article_singleTime.groupByKey().mapValues(orderedDateList)
  ordered_comments_by_date.takeOrdered(10)

Perform per user split to obtain training set, validation set and test set

In [ ]:
import math as m
#ceil so that the val always has more rows than test
#es. 6 items
#6 * 0.8 = 4.8
#6* 0.6 = 3.6
#with only in: [0:3] [3: 4] [4:6]
#with ceil on val: [0:3] [3:5] [5:6]
def splitEachUser(rdd_in):
  #1.0 added at the end representing the binary rating
  train_rdd = rdd_in.flatMap(lambda x:[(x[0],a,1.0) for a, _ in x[1][:int(len(x[1])*0.6)]])
  val_rdd = rdd_in.flatMap(lambda x:[(x[0],a,1.0) for a, _ in x[1][int(len(x[1])*0.6):m.ceil(len(x[1])*0.8)]])
  test_rdd = rdd_in.flatMap(lambda x:[(x[0],a,1.0) for a, _ in x[1][m.ceil(len(x[1])*0.8):]])

  return train_rdd, val_rdd, test_rdd

if not temporal, split randomly

In [ ]:
if temporal:
  [train_rdd, val_rdd, test_rdd] = splitEachUser(ordered_comments_by_date)
else:
  [train_rdd, val_rdd, test_rdd] = filtered_comments_article_singleTime.randomSplit([0.6,0.2,0.2], seed = 67)
  filtered_comments_article_singleTime.unpersist()
  train_rdd = train_rdd.map(lambda x: (x[0], x[1][0], 1.0))
  val_rdd   = val_rdd.map(lambda x: (x[0], x[1][0], 1.0))
  test_rdd = test_rdd.map(lambda x: (x[0], x[1][0], 1.0))

train_rdd.cache()
val_rdd.cache()
test_rdd.cache()

#train_rdd.takeOrdered(20)

PythonRDD[57] at RDD at PythonRDD.scala:56

In [ ]:
#val_rdd.takeOrdered(100)

In [ ]:
#test_rdd.takeOrdered(100)

HYPERPARAMETERS TUNING

Define our evaluation metric: we use precision@K, in particular precision@K

In [ ]:
import numpy as np
from pyspark.mllib.recommendation import ALS

K= 10

#create two broadcasted dictionaries:
seen_dict = sc.broadcast(train_rdd.map(lambda x: (x[0], x[1]) ).groupByKey().mapValues(set).collectAsMap()) #set of seen articles per user in training
test_dict = sc.broadcast(val_rdd.map(lambda x: (x[0], x[1]) ).groupByKey().mapValues(set).collectAsMap()) #set of test articles per user in test




Training loop with grid search

In [ ]:
#"""

if partial_data:
  ranks = [8, 16, 32]
else:
  ranks = [ 16, 32, 64]

alphas = [ 1.0, 10.0, 40.0]
best_rank = 8
best_alpha = 1.0
best_precision  = 0.0

def modelPrecision(model):
    V = model.productFeatures().collect() #list of (itemID, list(features))
    v_item_ids = [x[0] for x in V] #get the item_ids
    v_item_rows = sc.broadcast(np.array([x[1] for x in V])) #broadcast the feature vectors of the items (contained dimensions due to "low" rank (in our model 16k*64 max))
    v_item_map_id_idx = sc.broadcast({id: i for i, id in enumerate(v_item_ids)}) #broadcast dictionary that maps item_ids to their index in in v_item_ids
    v_item_ids = sc.broadcast(v_item_ids) #we broadcast the ids too
    def precisionAtKALS(user_row):
      #user_row example (u_id, user_features)
      u_id = user_row[0]
      u_lat_vec = np.array(user_row[1]) #np array for matrix multi

      test_arts = test_dict.value.get(u_id)
      if test_arts is None: #if there are no articles associated to that user in validation set
        return None
      seen_arts = seen_dict.value.get(u_id) #get the articles seen by the user in training, key always exist (in this code)
      #get broadcasts
      item_ids = v_item_ids.value
      item_rows = v_item_rows.value
      item_map_id_idx = v_item_map_id_idx.value

      scores = item_rows @ u_lat_vec #fast matrix multiplication to calculate scores

      seen_idx = [item_map_id_idx[id] for id in seen_arts if id in item_map_id_idx] #seen indexes of the score map
      if seen_idx:
          scores[seen_idx] = -np.inf #inf to facilitate sorting

      top_k_idx = np.argpartition(scores, -K)[-K:]
      top_k_arts = set(item_ids[i] for i in top_k_idx)

      precision = len(top_k_arts & test_arts)/ K

      return precision
    precisions_rdd = model.userFeatures().map(precisionAtKALS).filter(lambda x: x is not None)
    precision = precisions_rdd.mean()



    v_item_ids.unpersist()
    v_item_rows.unpersist()
    v_item_map_id_idx.unpersist()
    return precision
#hyperparameter tuning loop
for rank in ranks:
  for alpha in alphas:
    print('training with rank = {} and alpha = {}'.format(rank, alpha))
    model = ALS.trainImplicit(train_rdd, rank = rank, iterations = 10, lambda_ = 0.1, alpha= alpha, seed = 67)
    precision = modelPrecision(model)
    if precision > best_precision:
          best_precision = precision
          best_rank = rank
          best_alpha = alpha
    print('precision@10 = {}'.format(precision))

print('best alpha = {} best rank = {}  precision@10 = {}'.format(best_alpha, best_rank, best_precision))
test_dict.unpersist()
#"""

training with rank = 8 and alpha = 1.0
precision@10 = 0.0440188447357641
training with rank = 8 and alpha = 10.0
precision@10 = 0.04410077836952069
training with rank = 8 and alpha = 40.0
precision@10 = 0.04046838727297555
training with rank = 16 and alpha = 1.0
precision@10 = 0.04349310391915885
training with rank = 16 and alpha = 10.0
precision@10 = 0.04285811825754464
training with rank = 16 and alpha = 40.0
precision@10 = 0.03921207155537348
training with rank = 32 and alpha = 1.0


Train model

In [ ]:
#"""
#train model with best parameters
model = ALS.trainImplicit(train_rdd, rank = best_rank, iterations = 10, lambda_ = 0.1, alpha= best_alpha, seed = 67)
#"""

Test model on test set

In [ ]:
#calculate precision on test set
test_sets_rdd = test_rdd.map(lambda x: (x[0], x[1]) ).groupByKey().mapValues(set)
test_dict = sc.broadcast(test_sets_rdd.collectAsMap())
#"""
test_ALS_p_10 = modelPrecision(model)
print('precision@10 = {}'.format(test_ALS_p_10))
#"""

Baseline: popularity based



In [ ]:
popular_arts = train_rdd.map(lambda x: (x[1],1)).reduceByKey(lambda a,b: a+b).takeOrdered(10, key=lambda x: -x[1])
top_10_pop_art_set = set([x[0] for x in popular_arts])

def popularity_eval(test_set):
  return len(test_set & top_10_pop_art_set)/10

pop_precisions = test_sets_rdd.map(lambda x: popularity_eval(x[1])).collect()
pop_avg_precision = sum(pop_precisions)/len(pop_precisions)
print('popularityPrecision@10 = {}'.format(pop_avg_precision))



Read articles and convert to rdd

In [ ]:
articles_df = spark.read.csv('nyt-articles-2020.csv', header=True,inferSchema=True, multiLine=True, quote='"', sep=',', escape='"')
articles_rdd = articles_df.select(['uniqueID', 'keywords', 'headline', 'pub_date']).rdd.map(lambda x: (article_IDs_map.value.get(x[0]), x[1], x[2], x[3]))
articles_rdd.take(10)

In [ ]:
articles_rdd_filtered = articles_rdd.filter(lambda x: x[0] is not None and x[1] is not None)

articles_rdd_filtered.takeOrdered(10)

In [ ]:
articles_rdd_filtered.count()

CONTENT BASED MODEL

Constructing Item Profiles



In [ ]:
from collections import Counter

def tokenize(row):
  #row[1] = keywords
  keywords_unparsed = row[1]
  if keywords_unparsed is None:
    return None

  tokens = keywords_unparsed.strip("[]").replace('"','').replace("'",'').split(',')
  tokens = [x.strip() for x in tokens]

  return (row[0], Counter(tokens))

item_profile_rdd = articles_rdd_filtered.map(tokenize)
item_profile_rdd.take(3)


USER PROFILES

In [ ]:
joined_rdd = train_rdd.map(lambda x: (x[1], x[0])).join(item_profile_rdd.map(lambda x: (x[0], x[1])))
remapped_joined_rdd = joined_rdd.map(lambda x: (x[1][0], x[1][1]))
remapped_joined_rdd.take(10)

In [ ]:
user_profiles_rdd = remapped_joined_rdd.reduceByKey(lambda x,y: x + y)
user_profiles_rdd.take(2)

Function to perform l2-normalization to the profiles

In [ ]:
def normalize_profile(profile):
  sum_of_squares = sum(x**2 for x in profile.values())
  norm = m.sqrt(sum_of_squares)
  return {k: v/norm for k,v in profile.items()}
user_profile_normalized_rdd = user_profiles_rdd.mapValues(normalize_profile)
user_profile_normalized_rdd.take(2)

In [ ]:
item_profile_dict = sc.broadcast(item_profile_rdd.mapValues(normalize_profile).collectAsMap()) #broadcast the item_profiles as a map {item: normalized_profile}

Evaluation of CB

In [ ]:
#
def evaluate_precision_at_k(row):
  user_id = row[0]
  user_dict_word_weight = row[1] #normalized user_profile {keyword: weight}
  test_set = test_dict.value.get(user_id) #articles in test
  seen_items = seen_dict.value.get(user_id) #articles already seen in train

  precisions = []
  item_ids = []

  if test_set is None:
    return None
  user_set = set(user_dict_word_weight.keys()) #set of words in this user_profile for the intersection later
  for item, item_set in item_profile_dict.value.items(): #from the broadcasted item_profile dictionary
    intersection = item_set.keys() & user_set #intersection of user_set and item_set
    if len(intersection) == 0 or item in seen_items:
      continue
    score = sum(user_dict_word_weight[k] * item_set[k] for k in intersection ) #optimized dot product
    item_ids.append(item)
    precisions.append(score)
  item_ids_arr = np.array(item_ids)
  precisions_arr = np.array(precisions) #np for np.argpartition

  actual_K = min(len(precisions_arr), K) #to not have cases where there are less than K recommendations that would crash np.argpartition
  top_k_idx = np.argpartition(precisions_arr, -actual_K)[-actual_K:]
  top_k_items = set(item_ids_arr[top_k_idx])
  hits =  top_k_items & test_set
  precision_at_k = len(hits)/K
  return precision_at_k
evaluation_rdd = user_profile_normalized_rdd.map(evaluate_precision_at_k).filter(lambda x: x is not None)
print('precision@10 is: {}'.format(evaluation_rdd.mean()))